## Bronze layer

### Imports

In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F


# Market Data Structured

In [0]:
# ─────────────────────────────────────────
# structured_market_data
# ─────────────────────────────────────────

@dp.expect_or_drop("symbol_not_null", "symbol IS NOT NULL")
@dp.expect_or_drop("price_positive", "price > 0")
@dp.expect_or_drop("price_timestamp_not_null", "price_timestamp IS NOT NULL")
@dp.expect_or_drop("ingestion_timestamp_not_null", "ingestion_timestamp IS NOT NULL")
@dp.table(
    name    = "crypto_exchange.bronze.structured_market_data",
    comment = "Bronze — market data exploded to one row per coin per API call"
)
def structured_market_data():
    return (
        dp.read_stream("bronze_market_data")

        # Step 1: explode the data array so each coin becomes its own row
        .select(
            F.explode("data").alias("coin"),
            "endpoint",
            "ingestion_timestamp",
            "source",
            "year",
            "month",
            "day",
            "input_file_path",
            "ingest_timestamp"
        )

        # Step 2: flatten struct fields and cast types
        .select(
            F.col("coin.symbol").alias("symbol"),

            F.col("coin.price").alias("price"),

            F.col("coin.highest").alias("highest"),

            F.col("coin.lowest").alias("lowest"),

            F.col("coin.change_24h").alias("change_24h"),

            F.col("coin.market_cap").alias("market_cap"),

            F.col("coin.volume").alias("volume"),

            F.col("coin.source_exchange").alias("source_exchange"),

            F.to_timestamp(F.col("coin.date"), "yyyy-MM-dd HH:mm:ss").alias("price_timestamp"),

            F.col("endpoint"),

            F.to_timestamp(F.col("ingestion_timestamp")).alias("ingestion_timestamp"),
            
            F.col("source").alias("api_source"),
            
            F.col("year"),
            
            F.col("month"),
            
            F.col("day"),
            
            F.col("input_file_path"),
            
            F.col("ingest_timestamp")
        )
    )



### Conversions Data Structures

In [0]:
# ─────────────────────────────────────────
# structured_conversions
# ─────────────────────────────────────────

@dp.expect_or_drop("from_symbol_not_null", "from_symbol IS NOT NULL")
@dp.expect_or_drop("to_symbol_not_null", "to_symbol IS NOT NULL")
@dp.expect_or_drop("rate_positive", "rate > 0")
@dp.expect_or_drop("ingestion_timestamp_not_null", "ingestion_timestamp IS NOT NULL")
@dp.table(
    name    = "crypto_exchange.bronze.structured_conversions",
    comment = "Bronze — conversions data, one row per conversion pair per API call"
)
def structured_conversions():
    return (
        dp.read_stream("bronze_conversions")

        # Step 1: explode the data array so each conversion pair becomes its own row
        .select(
            F.explode("data").alias("conversion"),
            "endpoint",
            "ingestion_timestamp",
            "source",
            "year",
            "month",
            "day",
            "input_file_path",
            "ingest_timestamp"
        )

        # Step 2: flatten struct fields
        .select(
            F.col("conversion.from_symbol").alias("from_symbol"),

            F.col("conversion.to_symbol").alias("to_symbol"),

            F.col("conversion.amount").alias("amount"),

            F.col("conversion.converted_amount").alias("converted_amount"),

            F.col("conversion.rate").alias("rate"),

            F.col("endpoint"),

            F.to_timestamp(F.col("ingestion_timestamp")).alias("ingestion_timestamp"),

            F.col("source").alias("api_source"),

            F.col("year"),

            F.col("month"),

            F.col("day"),

            F.col("input_file_path"),

            F.col("ingest_timestamp")
        )
    )



### Crypto list Data Structured

In [0]:
# ─────────────────────────────────────────
# structured_crypto_list
# ─────────────────────────────────────────

@dp.expect_or_drop("symbol_not_null", "symbol IS NOT NULL")
@dp.expect_or_drop("id_not_null", "id IS NOT NULL")
@dp.expect_or_drop("ingestion_timestamp_not_null", "ingestion_timestamp IS NOT NULL")
@dp.table(
    name    = "crypto_exchange.bronze.structured_crypto_list",
    comment = "Bronze — full crypto list exploded to one row per coin"
)
def structured_crypto_list():
    return (
        dp.read_stream("bronze_crypto_list")

        # Step 1: explode the data array
        .select(
            F.explode("data").alias("coin"),
            "endpoint",
            "ingestion_timestamp",
            "source",
            "year",
            "month",
            "day",
            "input_file_path",
            "ingest_timestamp"
        )

        # Step 2: flatten struct fields and cast date strings to DateType
        .select(
            F.col("coin.id").alias("id"),
            
            F.col("coin.symbol").alias("symbol"),
            
            F.col("coin.name").alias("name"),
            
            F.col("coin.source").alias("source_exchange"),
            
            F.to_date(F.col("coin.history_available_from"), "yyyy-MM-dd").alias("history_available_from"),
            
            F.to_date(F.col("coin.ohlc_available_from"), "yyyy-MM-dd").alias("ohlc_available_from"),
            
            F.col("endpoint"),
            
            F.to_timestamp(F.col("ingestion_timestamp")).alias("ingestion_timestamp"),
            
            F.col("source").alias("api_source"),
            
            F.col("year"),
            
            F.col("month"),
            
            F.col("day"),
            
            F.col("input_file_path"),
            
            F.col("ingest_timestamp")
        )
    )